In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
from PIL import Image
from pathlib import Path

## Load Images

In [9]:
IMAGE_DIR = "Flickr8k dataset/Images"

In [10]:
caption_data = pd.read_csv("../data/Flickr8k dataset/captions.txt")
caption_data.head()

,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


## Generate Embeddings

In [ ]:
# download_model.py
from transformers import CLIPModel, CLIPProcessor

model_name = "openai/clip-vit-base-patch32"

CLIPModel.from_pretrained(model_name, cache_dir="./model")
CLIPProcessor.from_pretrained(model_name, cache_dir="./model")

In [11]:
# app.py
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

MODEL_PATH = "./model"
MODEL_PATH = "../backend/models"

# load once globally
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", cache_dir=MODEL_PATH)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", cache_dir=MODEL_PATH)

def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")

    # resize, centerCrop, totensor, normalize
    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        features = model.get_image_features(**inputs)
    
    # normalize (unit vector, for similarity)
    features = features / features.norm(p=2, dim=1, keepdim=True)

    return features.squeeze().tolist()

ValueError: Name tf.RaggedTensorSpec has already been registered for class tensorflow.python.ops.ragged.ragged_tensor.RaggedTensorSpec.

In [ ]:
images = [
    "../Flickr8k dataset/Images/667626_18933d713e.jpg",
    "../Flickr8k dataset/Images/23445819_3a458716c1.jpg",
    "../Flickr8k dataset/Images/10815824_2997e03d76.jpg",
    "../Flickr8k dataset/Images/44856031_0d82c2c7d1.jpg",
]

In [ ]:
def get_text_embedding(captions):
    # detect input type
    single_input = False
    
    if isinstance(captions, str):
        captions = [captions]
        single_input = True
    elif isinstance(captions, list) and len(captions) == 1:
        single_input = True

    inputs = processor(text=captions, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        text_features = model.get_text_features(**inputs)

    # normalize
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    embeddings = text_features.tolist()

    # return format control
    if single_input:
        return embeddings[0]   # single vector
    return embeddings          # list of vectors

In [ ]:
image_and_cap_embeddings = []
for image_path in images:
    image_id = image_path.split("/")[-1]
    print("processing...", image_id)

    embedding = get_image_embedding(image_path)
    
    caps = caption_data[caption_data['image'] == image_id]['caption'].tolist()
    cap_embeddings = get_text_embedding(caps)

    image_and_cap_embeddings.append((image_path.split("/")[-1], embedding, cap_embeddings))
    print("done")

In [ ]:
for id, image_em, caps in image_and_cap_embeddings:
    print(id)
    print("image_em:", image_em)
    for cap in caps:
        print("cap:", cap)

## Save Embeddings

In [ ]:
import os
IMAGE_DIR = '../Flickr8k dataset/Images'

image_paths = [
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.endswith((".jpg", ".png", ".jpeg"))
]

In [ ]:
image_paths[0].split('\\')

In [ ]:
import pickle

image_and_cap_embeddings = []
count = 0
for image_path in image_paths[:10]:
    image_id = image_path.split("\\")[-1]
    print("processing...", image_id)

    # image embedding
    img_emb = get_image_embedding(image_path)

    # captions list
    caps = caption_data[caption_data['image'] == image_id]['caption'].tolist()

    # build captions safely (NO mismatch possible)
    captions_data = []

    for cap in caps:
        emb = get_text_embedding(cap)  # single caption
        captions_data.append({
            "text": cap,
            "embedding": emb
        })

    # final record
    record = {
        "image_id": image_id,
        "image_url": image_id,
        "image_embedding": img_emb,
        "captions": captions_data
    }

    image_and_cap_embeddings.append(record)
    count += 1
        
print("done")

In [ ]:
# path = "./data/dataset.pkl"
with open("dataset.pkl", "wb") as f:
    pickle.dump(image_and_cap_embeddings, f)

In [ ]:
# path = "./data/dataset.pkl"
with open("dataset.pkl", "rb") as f:
    data = pickle.load(f)

### Safer version: Progressive save

In [ ]:
import pickle

data = []
last_image = None
SAVE_PATH = "dataset.pkl" # path = "./data/dataset.pkl"

# if os.path.exists(SAVE_PATH):
#     os.remove(SAVE_PATH)

try:
    for i, image_path in enumerate(image_paths[:100]):
        image_id = image_path.split("\\")[-1]
        last_image = image_id

        print("processing...", image_id)

        # image embedding
        img_emb = get_image_embedding(image_path)

        # captions list
        caps = caption_data[caption_data["image"] == image_id]["caption"].tolist()

        captions_data = []
        for cap in caps:
            emb = get_text_embedding(cap)
            captions_data.append({"text": cap, "embedding": emb})

        data.append(
            {
                "image_id": image_id,
                "image_url": image_id,
                "image_embedding": img_emb,
                "captions": captions_data,
            }
        )

        # save every 50
        if (i + 1) % 50 == 0:
            with open(SAVE_PATH, "wb") as f:
                pickle.dump(data, f)
            print("Saved at", i + 1)

    # final save
    with open(SAVE_PATH, "wb") as f:
        pickle.dump(data, f)

    print("done")

except Exception as e:
    print("failed at:", last_image)
    print("reason:", e)

    with open(SAVE_PATH, "wb") as f:
        pickle.dump(data, f)

    print("progress saved")

In [ ]:
import pickle
with open("dataset.pkl", "rb") as f: # path = "./data/dataset.pkl"
    data = pickle.load(f)

In [ ]:
print("Top-level fields in each record:")
for col, val in data[0].items():
    print(f"{col}: {type(val).__name__}")

print("\nCaption fields:")
for col, val in data[0]["captions"][0].items():
    print(f"{col}: {type(val).__name__}")

### Complete Pipeline (Batch Processing)

In [ ]:
import os, pickle
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import CLIPProcessor, CLIPModel
import torch

# ---------- CONFIG ----------
IMAGE_DIR = "../Flickr8k dataset/Images"
SAVE_PATH = "dataset.pkl" # path = "./data/dataset.pkl"
BATCH_SIZE = 16   # smaller for CPU
DEVICE = "cpu"

# ---------- MODEL ----------
# load once globally
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", cache_dir=MODEL_PATH)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", cache_dir=MODEL_PATH)

# ---------- DATA ----------
caption_data = pd.read_csv("../Flickr8k dataset/captions.txt")
cap_map = caption_data.groupby("image")["caption"].apply(list).to_dict()

image_paths = [
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
]

# ---------- FAST IMAGE LOADER ----------
def load_image(p):
    return Image.open(p).convert("RGB")

def load_images_parallel(paths):
    with ThreadPoolExecutor(max_workers=8):  # I/O speedup
        return list(map(load_image, paths))

# ---------- MAIN ----------
data = []

for i in range(0, len(image_paths), BATCH_SIZE):
    batch_paths = image_paths[i:i+BATCH_SIZE]
    image_ids = [os.path.basename(p) for p in batch_paths]

    print("batch:", i)

    # parallel image loading
    images = load_images_parallel(batch_paths)

    # batch image embedding
    inputs = processor(images=images, return_tensors="pt")
    with torch.no_grad():
        img_feats = model.get_image_features(**inputs)
    img_feats = img_feats.numpy()

    # batch text embedding
    all_caps = []
    cap_lens = []

    for img_id in image_ids:
        caps = cap_map.get(img_id, [])
        cap_lens.append(len(caps))
        all_caps.extend(caps)

    text_inputs = processor(text=all_caps, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        txt_feats = model.get_text_features(**text_inputs)
    txt_feats = txt_feats.numpy()

    # rebuild
    idx = 0
    for j, img_id in enumerate(image_ids):
        n = cap_lens[j]
        caps = cap_map.get(img_id, [])

        cap_embs = txt_feats[idx:idx+n]
        idx += n

        data.append({
            "image_id": img_id,
            "image_url": img_id,
            "image_embedding": img_feats[j],
            "captions": [
                {"text": c, "embedding": e}
                for c, e in zip(caps, cap_embs)
            ]
        })

    # save occasionally
    if (i // BATCH_SIZE) % 5 == 0:
        with open(SAVE_PATH, "wb") as f:
            pickle.dump(data, f)

# final save
with open(SAVE_PATH, "wb") as f:
    pickle.dump(data, f)

print("DONE ✅")